In [2]:
import sys
import numpy as np
import pandas as pd

print("PYTHON:", sys.executable)
print("numpy:", np.__version__, np.__file__)
print("pandas:", pd.__version__, pd.__file__)

PYTHON: /root/llm/je/bin/python
numpy: 2.2.6 /root/llm/je/lib/python3.10/site-packages/numpy/__init__.py
pandas: 2.3.3 /root/llm/je/lib/python3.10/site-packages/pandas/__init__.py


In [3]:
from pathlib import Path
import os
import sys
import subprocess
import json
import pandas as pd
from datetime import datetime

import torch

In [4]:
!pwd

/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/notebooks


In [5]:
path = "/root/llm/JOILang-Server"
#path = "/home/mgjeong/Desktop/llm/JOILang-Server"

In [6]:
py_path = "/root/llm/je/bin/python"
#py_path = "/home/mgjeong/miniconda3/envs/paper-gpu/bin/python"

In [7]:
# =============================================================================
# 0. Kernel / Python 환경 확인
# =============================================================================
print("=" * 100)
print("0. Kernel / Python 환경 확인")
print("KERNEL PYTHON:", sys.executable)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("torch cuda runtime:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "CUDA is not available in the current Jupyter kernel. "
        f"Kernel이 {py_path}인지 확인하세요."
    )

# resolve() 사용 금지: /root/llm/je/bin/python이 anaconda 원본으로 풀릴 수 있음
# /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
# a100 : "/root/llm/je/bin/python"
EXPECTED_PYTHON = os.path.abspath(py_path)
CURRENT_PYTHON = os.path.abspath(sys.executable)

if CURRENT_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        f"Wrong Jupyter kernel Python.\n"
        f"Expected: {EXPECTED_PYTHON}\n"
        f"Current : {CURRENT_PYTHON}\n"
        f"Jupyter에서 Kernel → Change Kernel → Python (/root/llm/je)로 바꾸세요."
    )


# =============================================================================
# 1. Repository / Script path 설정
# =============================================================================
print("=" * 100)
print("1. Repository / Script path 설정")
REPO = Path(path).absolute()
VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
RESULTS_ROOT = VERSION_DIR / "results"
LOCAL_MODEL_BASE = REPO / "local_models"

assert REPO.exists(), REPO
assert SCRIPT.exists(), SCRIPT

print("REPO:", REPO)
print("VERSION_DIR:", VERSION_DIR)
print("SCRIPT:", SCRIPT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)
print("LOCAL_MODEL_BASE exists:", LOCAL_MODEL_BASE.exists())


# =============================================================================
# 2. JOILang local model / worker 환경변수 설정
# =============================================================================
print("=" * 100)
print("2. JOILang local model / worker 환경변수 설정")
# 이전 실행에서 남아 있을 수 있는 충돌 변수 제거
for key in [
    "JOI_V15_LOCAL_MODEL_NAME",
    "JOI_V14_LOCAL_MODEL_NAME",
    "JOI_V14_WORKER_PYTHON",
    "JOI_V15_PERSISTENT_WORKER",   # 중요: persistent worker를 끄지 않기 위해 제거
]:
    os.environ.pop(key, None)

os.environ["PYTHONUNBUFFERED"] = "1"

# persistent worker는 유지하되, local model 위치만 지정
os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"

# worker도 현재 Jupyter kernel python과 동일하게 고정
os.environ["JOI_V15_WORKER_PYTHON"] = sys.executable

# debug
os.environ["JOI_V15_DEBUG_WORKER"] = "1"
os.environ["JOI_V15_DEBUG_LOG"] = "/tmp/joi_v15_worker_debug.log"

print("JOI_V15_WORKER_PYTHON:", os.environ.get("JOI_V15_WORKER_PYTHON"))
print("JOI_V15_LOCAL_MODEL_BASE_DIR:", os.environ.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
print("JOI_V15_LOCAL_DEVICE:", os.environ.get("JOI_V15_LOCAL_DEVICE"))
print("JOI_V15_LOCAL_FILES_ONLY:", os.environ.get("JOI_V15_LOCAL_FILES_ONLY"))
print("JOI_V15_PERSISTENT_WORKER:", os.environ.get("JOI_V15_PERSISTENT_WORKER"))
print("JOI_V15_DEBUG_LOG:", os.environ.get("JOI_V15_DEBUG_LOG"))


# =============================================================================
# 3. subprocess에서도 CUDA가 정상인지 확인
# =============================================================================
print("3. subprocess에서도 CUDA가 정상인지 확인")
subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import sys; "
            "print('subprocess python:', sys.executable); "
            "import torch; "
            "print('subprocess torch:', torch.__version__); "
            "print('subprocess cuda runtime:', torch.version.cuda); "
            "print('subprocess cuda available:', torch.cuda.is_available()); "
            "assert torch.cuda.is_available(), 'CUDA is not available in subprocess'; "
            "print('subprocess gpu:', torch.cuda.get_device_name(0))"
        ),
    ],
    check=True,
)

print("=" * 100)
print("Environment setup complete.")

0. Kernel / Python 환경 확인
KERNEL PYTHON: /root/llm/je/bin/python
pandas: 2.3.3
torch: 2.7.1+cu118
torch cuda runtime: 11.8
cuda available: True
GPU: NVIDIA A100 80GB PCIe
1. Repository / Script path 설정
REPO: /root/llm/JOILang-Server
VERSION_DIR: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413
SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py
RESULTS_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results
LOCAL_MODEL_BASE: /root/llm/JOILang-Server/local_models
LOCAL_MODEL_BASE exists: False
2. JOILang local model / worker 환경변수 설정
JOI_V15_WORKER_PYTHON: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_DEBUG_LOG: /tmp/joi_v15_worker_debug.log
3. subprocess에서도 CUDA가 정상인지 확인
subprocess python: /root/llm/je/bin/python
subprocess torch: 2.7.1+cu118
subprocess cuda runtime: 11.8
subpro

In [8]:
from pathlib import Path
import os
import sys
import subprocess
from datetime import datetime

# 서버별로 여기만 바꾸면 됨
path = "/root/llm/JOILang-Server"
# path = "/home/mgjeong/Desktop/llm/JOILang-Server"

py_path = "/root/llm/je/bin/python"
# py_path = "/home/mgjeong/miniconda3/envs/paper-gpu/bin/python"

REPO = Path(path).resolve()
SCRIPT = REPO / "gpt_mg/version0_15_update20260413/scripts/run_ga_search.py"
RESULTS_ROOT = REPO / "gpt_mg/version0_15_update20260413/results"
LOCAL_MODEL_BASE = REPO / "../local_models"

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
    "phi35_mini": "phi35_mini",
    "gemma2_9b_it": "gemma2_9b_it",
}

print("REPO:", REPO)
print("SCRIPT exists:", SCRIPT.exists())
print("RESULTS_ROOT:", RESULTS_ROOT)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE, LOCAL_MODEL_BASE.exists())
print("PYTHON:", py_path, Path(py_path).exists())

REPO: /root/llm/JOILang-Server
SCRIPT exists: True
RESULTS_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results
LOCAL_MODEL_BASE: /root/llm/JOILang-Server/../local_models True
PYTHON: /root/llm/je/bin/python True


In [9]:
for model_key, dirname in MODEL_DIRS.items():
    model_path = LOCAL_MODEL_BASE / dirname
    print("\n", model_key)
    print("path:", model_path)
    print("exists:", model_path.exists())
    if model_path.exists():
        print("config:", (model_path / "config.json").exists())
        print("tokenizer_config:", (model_path / "tokenizer_config.json").exists())
        print("tokenizer_json:", (model_path / "tokenizer.json").exists())
        print("safetensors_index:", (model_path / "model.safetensors.index.json").exists())


 qwen25_coder_7b
path: /root/llm/JOILang-Server/../local_models/qwen25_coder_7b
exists: True
config: True
tokenizer_config: True
tokenizer_json: True
safetensors_index: True

 llama31_8b
path: /root/llm/JOILang-Server/../local_models/llama31_8b
exists: True
config: True
tokenizer_config: True
tokenizer_json: True
safetensors_index: True

 qwen25_coder_14b
path: /root/llm/JOILang-Server/../local_models/qwen25_coder_14b
exists: True
config: True
tokenizer_config: True
tokenizer_json: True
safetensors_index: True

 phi35_mini
path: /root/llm/JOILang-Server/../local_models/phi35_mini
exists: True
config: True
tokenizer_config: True
tokenizer_json: True
safetensors_index: True

 gemma2_9b_it
path: /root/llm/JOILang-Server/../local_models/gemma2_9b_it
exists: True
config: True
tokenizer_config: True
tokenizer_json: True
safetensors_index: True


In [10]:
from pathlib import Path

# 서버별 repo 경로
path = "/root/llm/JOILang-Server"
# path = "/home/mgjeong/Desktop/llm/JOILang-Server"

# 서버별 python 경로
py_path = "/root/llm/je/bin/python"
# py_path = "/home/mgjeong/miniconda3/envs/paper-gpu/bin/python"

# 서버별 local model 경로: 실제 find 결과에 맞게 수정
local_model_base = "/root/llm/local_models"
# local_model_base = "/root/llm/JOILang-Server/local_models"
# local_model_base = "/home/mgjeong/Desktop/llm/local_models"

REPO = Path(path).resolve()
SCRIPT = REPO / "gpt_mg/version0_15_update20260413/scripts/run_ga_search.py"
RESULTS_ROOT = REPO / "gpt_mg/version0_15_update20260413/results"
LOCAL_MODEL_BASE = Path(local_model_base).resolve()

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
    "phi35_mini": "phi35_mini",
    "gemma2_9b_it": "gemma2_9b_it",
}

print("REPO:", REPO, REPO.exists())
print("SCRIPT:", SCRIPT, SCRIPT.exists())
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE, LOCAL_MODEL_BASE.exists())

REPO: /root/llm/JOILang-Server True
SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py True
LOCAL_MODEL_BASE: /root/llm/local_models True


In [ ]:
for model_key, dirname in MODEL_DIRS.items():
    model_path = LOCAL_MODEL_BASE / dirname
    print("\n", model_key)
    print("path:", model_path)
    print("exists:", model_path.exists())
    if model_path.exists():
        print("config:", (model_path / "config.json").exists())
        print("tokenizer_config:", (model_path / "tokenizer_config.json").exists())
        print("tokenizer_json:", (model_path / "tokenizer.json").exists())
        print("safetensors_index:", (model_path / "model.safetensors.index.json").exists())

In [ ]:
from pathlib import Path

local_model_base = "/root/llm/local_models"
LOCAL_MODEL_BASE = Path(local_model_base)

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
}

for model_key, dirname in MODEL_DIRS.items():
    p = LOCAL_MODEL_BASE / dirname
    print("\n", model_key)
    print("path:", p)
    print("exists:", p.exists())
    if p.exists():
        print("config:", (p / "config.json").exists())
        print("tokenizer:", (p / "tokenizer.json").exists())
        print("index:", (p / "model.safetensors.index.json").exists())

# 기본 함수 정의

In [ ]:
# ============================================================
# DEFINE run_ga_all_categories() WRAPPER FOR NEW JUPYTER KERNEL
# ============================================================

from pathlib import Path
from datetime import datetime
import os
import sys
import subprocess
import time

# ------------------------------------------------------------
# A100 server paths
# ------------------------------------------------------------
SERVER_PRESET = "A100_SET_B"

REPO = Path("/root/llm/JOILang-Server").resolve()
PY = "/root/llm/je/bin/python"
VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
RESULTS_ROOT = VERSION_DIR / "results"
LOCAL_MODEL_BASE = Path("/root/llm/local_models").resolve()

MODEL_LOCAL_DIRS = {
    "qwen25_coder_7b": LOCAL_MODEL_BASE / "qwen25_coder_7b",
    "llama31_8b": LOCAL_MODEL_BASE / "llama31_8b",
    "qwen25_coder_14b": LOCAL_MODEL_BASE / "qwen25_coder_14b",
    "phi35_mini": LOCAL_MODEL_BASE / "phi35_mini",
    "gemma2_9b_it": LOCAL_MODEL_BASE / "gemma2_9b_it",
}

assert REPO.exists(), REPO
assert Path(PY).exists(), PY
assert SCRIPT.exists(), SCRIPT
assert RESULTS_ROOT.exists(), RESULTS_ROOT
assert LOCAL_MODEL_BASE.exists(), LOCAL_MODEL_BASE

# ------------------------------------------------------------
# Detect supported CLI flags
# ------------------------------------------------------------
try:
    HELP_TEXT = subprocess.check_output(
        [PY, str(SCRIPT), "--help"],
        cwd=str(REPO),
        text=True,
        stderr=subprocess.STDOUT,
        timeout=60,
    )
except Exception as e:
    print("[WARN] Could not read run_ga_search.py --help:", repr(e))
    HELP_TEXT = ""

def supports_flag(flag: str) -> bool:
    return flag in HELP_TEXT

def add_arg(cmd, flag, value=None):
    if supports_flag(flag):
        cmd.append(flag)
        if value is not None:
            cmd.append(str(value))
    else:
        print(f"[SKIP unsupported flag] {flag}")

def add_bool(cmd, flag, enabled=True):
    if not enabled:
        return
    if supports_flag(flag):
        cmd.append(flag)
    else:
        print(f"[SKIP unsupported flag] {flag}")

# ------------------------------------------------------------
# Main wrapper
# ------------------------------------------------------------
def run_ga_all_categories(
    model_key: str,
    categories=(1, 2, 3, 4, 5, 6, 7, 8),
    limit_per_category=3,
    sample_size=24,
    validation_size=24,
    population=5,
    gens=10,
    target_detpass=90,
    base_prefix="ga_A100_SET_B",
    use_advisor=False,
    full_run=True,
    progress="verbose",
    timeout_sec=1200,
    retries=0,
    idle_timeout_sec=None,
    total_timeout_sec=None,

    # advisor
    advisor_trigger_mode="always",
    advisor_min_population_for_child=4,
    advisor_force_child_quota=True,
    use_mock_advisor=False,
    advisor_model_key="gpt41_mini",

    # compression-aware flags
    compression_detpass_threshold=90,
    aggressive_compression_after_target=True,
    compression_child_quota=1,
    compression_child_ratio=0.2,
    advisor_compression_child_quota=1,
    advisor_prefer_compression_after_detpass=90,
    compression_token_reduction_target=0.15,
    compression_token_plateau_delta=1.0,
    allow_aggressive_compression=True,
):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    cat_text = "".join(str(c) for c in categories)
    mode = "cloud_advisor" if use_advisor else "cloudless"

    model_path = MODEL_LOCAL_DIRS.get(model_key)
    if model_path is None:
        raise KeyError(f"Unknown model_key: {model_key}")

    model_realpath = Path(model_path).resolve()

    if not model_realpath.exists():
        raise FileNotFoundError(f"Local model path does not exist: {model_realpath}")

    out_dir = (
        RESULTS_ROOT
        / f"{base_prefix}_{mode}_cat{cat_text}_lpc{limit_per_category}"
          f"_pop{population}_gens{gens}_{model_key}_{ts}"
        / "ga_output"
    )

    cmd = [
        PY,
        "-u",
        str(SCRIPT),
        "--profile", "version0_15",
        "--model-key", model_key,
        "--target-detpass", str(target_detpass),
    ]

    # LLM mode
    if use_mock_advisor:
        add_arg(cmd, "--llm-mode", "mock")
    else:
        add_arg(cmd, "--llm-mode", "worker")

    # GA base config
    add_arg(cmd, "--population", population)
    add_arg(cmd, "--gens", gens)
    add_arg(cmd, "--min-generations", gens)
    add_arg(cmd, "--max-generations", gens)
    add_arg(cmd, "--sample-size", sample_size)
    add_arg(cmd, "--validation-size", validation_size)
    add_arg(cmd, "--cheap-eval-limit", 2)
    add_arg(cmd, "--candidate-k", 1)
    add_arg(cmd, "--repair-attempts", 0)
    add_arg(cmd, "--det-profile", "strict")
    add_bool(cmd, "--feedback-guided-mutation", True)

    # redesign config
    add_arg(cmd, "--selection-mode", "redesign")
    add_arg(cmd, "--fitness-mode", "phase_aware")
    add_arg(cmd, "--mutation-mode", "cloudless_decompiler")
    add_bool(cmd, "--enable-compression-mutation", True)
    add_bool(cmd, "--enable-prompt-decompiler", True)
    add_bool(cmd, "--enable-rendered-prompt-dedupe", True)
    add_bool(cmd, "--enable-pareto-archive", True)
    add_bool(cmd, "--enable-group-specialist-archives", True)

    add_arg(cmd, "--category-balance-mode", "guard")
    add_arg(cmd, "--token-penalty-mode", "hybrid")
    add_arg(cmd, "--stop-controller-mode", "active")
    add_arg(cmd, "--plateau-window", 1)
    add_arg(cmd, "--disruptive-max-attempts", 1)
    add_arg(cmd, "--reasoning-mutation-mode", "auto")
    add_arg(cmd, "--intent-hint-mode", "auto")

    # progress / timeout / output
    add_arg(cmd, "--progress", progress)
    add_arg(cmd, "--timeout-sec", timeout_sec)
    add_arg(cmd, "--retries", retries)
    if idle_timeout_sec is not None:
        add_arg(cmd, "--idle-timeout-sec", idle_timeout_sec)
    if total_timeout_sec is not None:
        add_arg(cmd, "--total-timeout-sec", total_timeout_sec)

    add_arg(cmd, "--limit-per-category", limit_per_category)
    add_arg(cmd, "--output-root", out_dir)

    if full_run:
        add_bool(cmd, "--full-run", True)

    for c in categories:
        add_arg(cmd, "--category", c)

    # advisor config
    if use_advisor:
        if not use_mock_advisor and not os.environ.get("OPENAI_API_KEY", "").strip():
            raise RuntimeError("OPENAI_API_KEY is not set. Set it before real cloud-advisor run.")

        add_bool(cmd, "--llm-mutation-advisor", True)
        add_arg(cmd, "--advisor-model-key", advisor_model_key)
        add_arg(cmd, "--advisor-trigger-mode", advisor_trigger_mode)
        add_arg(cmd, "--advisor-min-population-for-child", advisor_min_population_for_child)
        add_bool(cmd, "--advisor-force-child-quota", advisor_force_child_quota)
    else:
        add_arg(cmd, "--advisor-trigger-mode", "off")

    # compression-aware config
    add_arg(cmd, "--compression-detpass-threshold", compression_detpass_threshold)
    add_bool(cmd, "--aggressive-compression-after-target", aggressive_compression_after_target)
    add_arg(cmd, "--compression-child-quota", compression_child_quota)
    add_arg(cmd, "--compression-child-ratio", compression_child_ratio)
    add_arg(cmd, "--advisor-compression-child-quota", advisor_compression_child_quota)
    add_arg(cmd, "--advisor-prefer-compression-after-detpass", advisor_prefer_compression_after_detpass)
    add_arg(cmd, "--compression-token-reduction-target", compression_token_reduction_target)
    add_arg(cmd, "--compression-token-plateau-delta", compression_token_plateau_delta)
    add_bool(cmd, "--allow-aggressive-compression", allow_aggressive_compression)

    run_env = os.environ.copy()
    run_env["JOI_V15_WORKER_PYTHON"] = PY
    run_env["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
    run_env["JOI_V15_LOCAL_MODEL_NAME"] = str(model_realpath)
    run_env["JOI_V15_LOCAL_DEVICE"] = run_env.get("JOI_V15_LOCAL_DEVICE", "cuda:0")
    run_env["JOI_V15_LOCAL_FILES_ONLY"] = "true"
    run_env["TRANSFORMERS_OFFLINE"] = "1"
    run_env["HF_HUB_OFFLINE"] = "1"

    debug_log = Path("/tmp") / f"joi_v15_worker_debug_{model_key}_{mode}_{ts}.log"
    run_env["JOI_V15_WORKER_DEBUG_LOG"] = str(debug_log)

    print("=" * 120)
    print(f"RUN: {model_key} / {mode}")
    print("OUTPUT:", out_dir)
    print("REPO:", REPO)
    print("PY:", PY)
    print("JOI_V15_LOCAL_MODEL_BASE_DIR:", run_env["JOI_V15_LOCAL_MODEL_BASE_DIR"])
    print("JOI_V15_LOCAL_MODEL_NAME:", run_env["JOI_V15_LOCAL_MODEL_NAME"])
    print("JOI_V15_LOCAL_DEVICE:", run_env["JOI_V15_LOCAL_DEVICE"])
    print("DEBUG_LOG:", debug_log)
    print("COMMAND:")
    print(" ".join(map(str, cmd)))
    print("=" * 120)

    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(REPO),
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line, end="")

    rc = proc.wait()

    print("\nRETURN CODE:", rc)
    print("OUTPUT:", out_dir)
    print("DEBUG_LOG:", debug_log)

    if rc != 0:
        raise RuntimeError(f"run_ga_all_categories failed with return code {rc}")

    return out_dir

print("[OK] run_ga_all_categories is now defined.")
print("Signature supports compression-aware kwargs.")

# Cell 1. 환경/수정 반영/테스트 확인

In [ ]:
from pathlib import Path
import os
import re
import json
import time
import getpass
import inspect
import subprocess
import traceback
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 0. A100 server config
# ============================================================

SERVER_PRESET = "A100_SET_B"

REPO = Path("/root/llm/JOILang-Server")
PY = "/root/llm/je/bin/python"
VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
RESULTS_ROOT = VERSION_DIR / "results"
LOCAL_MODEL_BASE = Path("/root/llm/local_models")

print("=" * 120)
print("[CONFIG]")
print("REPO:", REPO, REPO.exists())
print("PY:", PY, Path(PY).exists())
print("SCRIPT:", SCRIPT, SCRIPT.exists())
print("RESULTS_ROOT:", RESULTS_ROOT, RESULTS_ROOT.exists())
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE, LOCAL_MODEL_BASE.exists())
print("=" * 120)

assert REPO.exists()
assert Path(PY).exists()
assert SCRIPT.exists()
assert RESULTS_ROOT.exists()
assert LOCAL_MODEL_BASE.exists()

# ============================================================
# 1. OpenAI API key
# ============================================================

if not os.environ.get("OPENAI_API_KEY", "").strip():
    key = getpass.getpass("OPENAI_API_KEY: ").strip()
    if not key:
        raise RuntimeError("OPENAI_API_KEY is required for real cloud-advisor test.")
    os.environ["OPENAI_API_KEY"] = key
    print("[OK] OPENAI_API_KEY set in notebook process.")
else:
    print("[OK] OPENAI_API_KEY already set.")

# ============================================================
# 2. Required flags / code keywords check
# ============================================================

src = SCRIPT.read_text(encoding="utf-8")

required_keywords = [
    "--compression-detpass-threshold",
    "--aggressive-compression-after-target",
    "--compression-child-quota",
    "--compression-child-ratio",
    "--advisor-compression-child-quota",
    "--advisor-prefer-compression-after-detpass",
    "--compression-token-reduction-target",
    "--compression-token-plateau-delta",
    "--allow-aggressive-compression",
    "on_compression",
    "COMPRESSION_SEARCH",
    "switch_aggressive_compression",
    "advisor_compression_children_scheduled",
    "cloudless_compression_fallback_scheduled",
]

missing = [k for k in required_keywords if k not in src]
print("\n[REQUIRED KEYWORDS]")
if missing:
    print("[FAIL] missing:", missing)
    raise RuntimeError(f"Compression redesign keywords missing: {missing}")
else:
    print("[OK] all required compression keywords exist.")

if "run_ga_all_categories" not in globals():
    raise RuntimeError("run_ga_all_categories is not defined. Run the notebook cell defining wrapper first.")

sig = inspect.signature(run_ga_all_categories)
wrapper_required_kwargs = [
    "compression_detpass_threshold",
    "aggressive_compression_after_target",
    "compression_child_quota",
    "compression_child_ratio",
    "advisor_compression_child_quota",
    "advisor_prefer_compression_after_detpass",
    "compression_token_reduction_target",
    "compression_token_plateau_delta",
    "allow_aggressive_compression",
]
missing_wrapper = [k for k in wrapper_required_kwargs if k not in sig.parameters]
print("\n[WRAPPER SIGNATURE]")
if missing_wrapper:
    print("[FAIL] run_ga_all_categories missing kwargs:", missing_wrapper)
    raise RuntimeError(f"Notebook wrapper not updated: {missing_wrapper}")
else:
    print("[OK] run_ga_all_categories supports compression kwargs.")

# ============================================================
# 3. Model path check
# ============================================================

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
}

model_rows = []
for model_key, dirname in MODEL_DIRS.items():
    p = LOCAL_MODEL_BASE / dirname
    model_rows.append({
        "model_key": model_key,
        "path": str(p),
        "exists": p.exists(),
        "config": (p / "config.json").exists(),
        "tokenizer": (p / "tokenizer.json").exists(),
        "index": (p / "model.safetensors.index.json").exists(),
        "safetensors_count": len(list(p.glob("*.safetensors"))) if p.exists() else 0,
    })

model_df = pd.DataFrame(model_rows)
display(model_df)

assert model_df["exists"].all()
assert model_df["config"].all()
assert model_df["tokenizer"].all()
assert model_df["index"].all()
assert (model_df["safetensors_count"] > 0).all()

# ============================================================
# 4. compile / optional pytest
# ============================================================

cmds = [
    [PY, "-m", "compileall", str(VERSION_DIR / "scripts")],
    [PY, "-m", "compileall", str(VERSION_DIR / "tests")],
]

test_files = [
    VERSION_DIR / "tests/test_compression_mutation_policy.py",
    VERSION_DIR / "tests/test_ga_redesign.py",
    VERSION_DIR / "tests/test_advisor_feedback_loop.py",
]

for cmd in cmds:
    print("\nRUN:", " ".join(map(str, cmd)))
    subprocess.check_call(cmd, cwd=str(REPO))

for test_file in test_files:
    if test_file.exists():
        cmd = [PY, "-m", "pytest", "-q", str(test_file)]
        print("\nRUN:", " ".join(map(str, cmd)))
        try:
            subprocess.check_call(cmd, cwd=str(REPO))
        except subprocess.CalledProcessError as e:
            print("[WARN] pytest failed or pytest is not installed:", e)

print("\n[OK] environment / code / compile checks done.")

# Cell 2. Compression smoke test: cloudless fallback + cloud advisor

In [ ]:
# ============================================================
# Compression-aware smoke tests
# ============================================================

COMMON_COMPRESSION_KWARGS = dict(
    compression_detpass_threshold=90,
    aggressive_compression_after_target=True,
    compression_child_quota=1,
    compression_child_ratio=0.25,
    advisor_compression_child_quota=1,
    advisor_prefer_compression_after_detpass=90,
    compression_token_reduction_target=0.15,
    compression_token_plateau_delta=1.0,
    allow_aggressive_compression=True,
)

def inspect_compression_artifacts(out_dir, run_name):
    out_dir = Path(out_dir)
    summary_path = out_dir / "ga_summary.json"
    progress_path = out_dir / "ga_generation_progress.csv"
    trans_path = out_dir / "population_transitions.csv"
    proposals_path = out_dir / "mutation_proposals.jsonl"

    row = {
        "run": run_name,
        "out_dir": str(out_dir),
        "summary_exists": summary_path.exists(),
        "progress_exists": progress_path.exists(),
        "transition_exists": trans_path.exists(),
        "proposals_exists": proposals_path.exists(),
        "best_DETPass": np.nan,
        "best_so_far_DETPass": np.nan,
        "avg_prompt_tokens": np.nan,
        "tokens_gt_0": False,
        "compression_ready_final": None,
        "compression_success_count": 0,
        "compression_rejection_count": 0,
        "advisor_compression_proposals_generated": 0,
        "advisor_compression_children_scheduled": 0,
        "cloudless_compression_fallback_scheduled": 0,
        "new_by_compression_sum": 0,
        "new_by_advisor_sum": 0,
        "compression_proposal_count": 0,
        "advisor_proposal_count": 0,
    }

    if summary_path.exists():
        s = json.loads(summary_path.read_text(encoding="utf-8"))
        for k in [
            "best_DETPass",
            "best_so_far_DETPass",
            "compression_ready_final",
            "compression_success_count",
            "compression_rejection_count",
            "advisor_compression_proposals_generated",
            "advisor_compression_children_scheduled",
            "cloudless_compression_fallback_scheduled",
            "compact_best_tokens",
            "compact_best_DETPass",
        ]:
            if k in s:
                row[k] = s.get(k)

    if progress_path.exists():
        df = pd.read_csv(progress_path)
        if len(df):
            if "avg_prompt_tokens" in df.columns:
                row["avg_prompt_tokens"] = pd.to_numeric(df["avg_prompt_tokens"], errors="coerce").max()
                row["tokens_gt_0"] = bool(row["avg_prompt_tokens"] > 0)
            if "best_so_far_DETPass" in df.columns:
                row["best_so_far_DETPass"] = pd.to_numeric(df["best_so_far_DETPass"], errors="coerce").max()

    if trans_path.exists():
        tdf = pd.read_csv(trans_path)
        for col, key in [
            ("new_by_compression", "new_by_compression_sum"),
            ("new_by_advisor", "new_by_advisor_sum"),
            ("advisor_compression_children_scheduled", "advisor_compression_children_scheduled"),
            ("cloudless_compression_fallback_scheduled", "cloudless_compression_fallback_scheduled"),
            ("advisor_compression_proposals", "advisor_compression_proposals_generated"),
        ]:
            if col in tdf.columns:
                row[key] = int(pd.to_numeric(tdf[col], errors="coerce").fillna(0).sum())

    if proposals_path.exists():
        for line in proposals_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue
            source = str(obj.get("source", ""))
            family = str(obj.get("mutation_family", ""))
            op = str(obj.get("operator", ""))

            if source == "advisor":
                row["advisor_proposal_count"] += 1
            if family == "compression" or "compress" in op or "token" in op or "few_shot" in op:
                row["compression_proposal_count"] += 1

    row["compression_child_or_proposal_ok"] = (
        row["compression_proposal_count"] > 0
        or row["new_by_compression_sum"] > 0
        or row["advisor_compression_children_scheduled"] > 0
        or row["cloudless_compression_fallback_scheduled"] > 0
    )

    row["ok"] = bool(
        row["summary_exists"]
        and row["progress_exists"]
        and row["tokens_gt_0"]
        and row["compression_child_or_proposal_ok"]
    )
    return row


smoke_runs = {}
smoke_rows = []

# ------------------------------------------------------------
# 1. Cloudless fallback compression smoke
# ------------------------------------------------------------

try:
    out = run_ga_all_categories(
        model_key="qwen25_coder_14b",
        categories=(1,),
        limit_per_category=1,
        sample_size=1,
        validation_size=1,
        population=4,
        gens=3,
        target_detpass=90,
        base_prefix=f"smoke_compression_cloudless_{SERVER_PRESET}",
        use_advisor=False,
        full_run=True,
        progress="verbose",
        timeout_sec=1200,
        retries=0,
        advisor_trigger_mode="off",
        advisor_min_population_for_child=4,
        advisor_force_child_quota=False,
        use_mock_advisor=False,
        **COMMON_COMPRESSION_KWARGS,
    )
    smoke_runs["14B_cloudless_compression"] = out
    smoke_rows.append(inspect_compression_artifacts(out, "14B_cloudless_compression"))
except Exception as e:
    traceback.print_exc()
    smoke_runs["14B_cloudless_compression"] = None
    smoke_rows.append({"run": "14B_cloudless_compression", "ok": False, "error": repr(e)})

# ------------------------------------------------------------
# 2. Real cloud advisor compression smoke
# ------------------------------------------------------------

try:
    out = run_ga_all_categories(
        model_key="qwen25_coder_14b",
        categories=(1,),
        limit_per_category=1,
        sample_size=1,
        validation_size=1,
        population=4,
        gens=3,
        target_detpass=90,
        base_prefix=f"smoke_compression_advisor_{SERVER_PRESET}",
        use_advisor=True,
        full_run=True,
        progress="verbose",
        timeout_sec=1200,
        retries=0,
        advisor_trigger_mode="always",
        advisor_min_population_for_child=4,
        advisor_force_child_quota=True,
        use_mock_advisor=False,
        **COMMON_COMPRESSION_KWARGS,
    )
    smoke_runs["14B_cloud_advisor_compression"] = out
    smoke_rows.append(inspect_compression_artifacts(out, "14B_cloud_advisor_compression"))
except Exception as e:
    traceback.print_exc()
    smoke_runs["14B_cloud_advisor_compression"] = None
    smoke_rows.append({"run": "14B_cloud_advisor_compression", "ok": False, "error": repr(e)})

smoke_df = pd.DataFrame(smoke_rows)
display(smoke_df)

failed = smoke_df[smoke_df["ok"] != True]
if len(failed):
    raise RuntimeError("Compression smoke failed. Check displayed rows.")

print("[OK] Compression smoke tests passed.")
smoke_runs

In [ ]:
# ============================================================
# HOTFIX: _crossover must handle compressed candidate_strategies length <= 1
# ============================================================

from pathlib import Path
import re
import subprocess

REPO = Path("/root/llm/JOILang-Server")
PY = "/root/llm/je/bin/python"
SCRIPT = REPO / "gpt_mg/version0_15_update20260413/scripts/run_ga_search.py"

src = SCRIPT.read_text(encoding="utf-8")

marker = "Compression may reduce candidate_strategies to a single safe strategy."

if marker in src:
    print("[OK] _crossover candidate_strategies hotfix already exists.")
else:
    pattern = r'^(?P<indent>\s*)child\["params"\]\["candidate_strategies"\]\s*=\s*strategies\[:\s*rng\.randint\(2,\s*min\(5,\s*len\(strategies\)\)\)\]'

    def repl(m):
        indent = m.group("indent")
        return "\n".join([
            f'{indent}# {marker}',
            f'{indent}strategies = list(strategies or [])',
            f'{indent}if len(strategies) == 0:',
            f'{indent}    child["params"]["candidate_strategies"] = ["minimal"]',
            f'{indent}elif len(strategies) == 1:',
            f'{indent}    child["params"]["candidate_strategies"] = strategies',
            f'{indent}else:',
            f'{indent}    upper = min(5, len(strategies))',
            f'{indent}    child["params"]["candidate_strategies"] = strategies[: rng.randint(2, upper)]',
        ])

    new_src, n = re.subn(pattern, repl, src, count=1, flags=re.M)

    if n != 1:
        raise RuntimeError("Failed to patch _crossover candidate_strategies randint line.")

    SCRIPT.write_text(new_src, encoding="utf-8")
    print("[PATCHED] _crossover now supports candidate_strategies length 0/1.")

subprocess.check_call([PY, "-m", "py_compile", str(SCRIPT)], cwd=str(REPO))
print("[OK] py_compile passed.")

## 기존: 예전 약한 artifact check 버전

In [18]:
# ============================================================
# RETRY ONLY: real cloud advisor compression smoke
# ============================================================

from pathlib import Path
import json
import traceback
import pandas as pd
import numpy as np

def inspect_compression_artifacts_simple(out_dir, run_name):
    out_dir = Path(out_dir)

    summary_path = out_dir / "ga_summary.json"
    progress_path = out_dir / "ga_generation_progress.csv"
    trans_path = out_dir / "population_transitions.csv"
    proposals_path = out_dir / "mutation_proposals.jsonl"

    row = {
        "run": run_name,
        "out_dir": str(out_dir),
        "summary_exists": summary_path.exists(),
        "progress_exists": progress_path.exists(),
        "transition_exists": trans_path.exists(),
        "proposals_exists": proposals_path.exists(),
        "best_DETPass": np.nan,
        "best_so_far_DETPass": np.nan,
        "avg_prompt_tokens": np.nan,
        "tokens_gt_0": False,
        "compression_proposal_count": 0,
        "advisor_proposal_count": 0,
        "new_by_compression_sum": 0,
        "new_by_advisor_sum": 0,
        "advisor_compression_children_scheduled": 0,
        "cloudless_compression_fallback_scheduled": 0,
        "ok": False,
    }

    if summary_path.exists():
        s = json.loads(summary_path.read_text(encoding="utf-8"))
        row["best_DETPass"] = s.get("best_DETPass", s.get("best_so_far_DETPass"))
        row["best_so_far_DETPass"] = s.get("best_so_far_DETPass")
        row["advisor_compression_children_scheduled"] = s.get("advisor_compression_children_scheduled", 0)
        row["cloudless_compression_fallback_scheduled"] = s.get("cloudless_compression_fallback_scheduled", 0)

    if progress_path.exists():
        df = pd.read_csv(progress_path)
        if len(df) and "avg_prompt_tokens" in df.columns:
            row["avg_prompt_tokens"] = pd.to_numeric(df["avg_prompt_tokens"], errors="coerce").max()
            row["tokens_gt_0"] = bool(row["avg_prompt_tokens"] > 0)
        if len(df) and "best_so_far_DETPass" in df.columns:
            row["best_so_far_DETPass"] = pd.to_numeric(df["best_so_far_DETPass"], errors="coerce").max()

    if trans_path.exists():
        tdf = pd.read_csv(trans_path)

        if "new_by_compression" in tdf.columns:
            row["new_by_compression_sum"] = int(pd.to_numeric(tdf["new_by_compression"], errors="coerce").fillna(0).sum())

        if "new_by_advisor" in tdf.columns:
            row["new_by_advisor_sum"] = int(pd.to_numeric(tdf["new_by_advisor"], errors="coerce").fillna(0).sum())

        if "advisor_compression_children_scheduled" in tdf.columns:
            row["advisor_compression_children_scheduled"] = int(
                pd.to_numeric(tdf["advisor_compression_children_scheduled"], errors="coerce").fillna(0).sum()
            )

        if "cloudless_compression_fallback_scheduled" in tdf.columns:
            row["cloudless_compression_fallback_scheduled"] = int(
                pd.to_numeric(tdf["cloudless_compression_fallback_scheduled"], errors="coerce").fillna(0).sum()
            )

    if proposals_path.exists():
        for line in proposals_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue

            source = str(obj.get("source", ""))
            family = str(obj.get("mutation_family", ""))
            op = str(obj.get("operator", ""))

            if source == "advisor":
                row["advisor_proposal_count"] += 1

            if family == "compression" or "compress" in op or "token" in op or "few_shot" in op:
                row["compression_proposal_count"] += 1

    row["compression_ok"] = (
        row["compression_proposal_count"] > 0
        or row["new_by_compression_sum"] > 0
        or row["advisor_compression_children_scheduled"] > 0
        or row["cloudless_compression_fallback_scheduled"] > 0
    )

    row["ok"] = bool(
        row["summary_exists"]
        and row["progress_exists"]
        and row["tokens_gt_0"]
        and row["compression_ok"]
    )

    return row


try:
    advisor_retry_out = run_ga_all_categories(
        model_key="qwen25_coder_14b",
        categories=(1,),
        limit_per_category=1,
        sample_size=1,
        validation_size=1,
        population=4,
        gens=3,
        target_detpass=90,
        base_prefix=f"smoke_compression_advisor_retry_{SERVER_PRESET}",
        use_advisor=True,
        full_run=True,
        progress="verbose",
        timeout_sec=1200,
        retries=0,

        advisor_trigger_mode="always",
        advisor_min_population_for_child=4,
        advisor_force_child_quota=True,
        use_mock_advisor=False,

        compression_detpass_threshold=90,
        aggressive_compression_after_target=True,
        compression_child_quota=1,
        compression_child_ratio=0.25,
        advisor_compression_child_quota=1,
        advisor_prefer_compression_after_detpass=90,
        compression_token_reduction_target=0.15,
        compression_token_plateau_delta=1.0,
        allow_aggressive_compression=True,
    )

    advisor_retry_row = inspect_compression_artifacts_simple(
        advisor_retry_out,
        "14B_cloud_advisor_compression_retry"
    )

    advisor_retry_df = pd.DataFrame([advisor_retry_row])
    display(advisor_retry_df)

    if not advisor_retry_row["ok"]:
        raise RuntimeError("Advisor compression retry did not pass artifact checks.")

    print("[OK] Cloud advisor compression retry passed.")
    print("advisor_retry_out =", advisor_retry_out)

except Exception as e:
    traceback.print_exc()
    raise
    

RUN: qwen25_coder_14b / cloud_advisor
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_compression_advisor_retry_A100_SET_B_cloud_advisor_cat1_lpc1_pop4_gens3_qwen25_coder_14b_20260611_105957/ga_output
REPO: /root/llm/JOILang-Server
PY: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/local_models
JOI_V15_LOCAL_MODEL_NAME: /root/llm/local_models/qwen25_coder_14b
JOI_V15_LOCAL_DEVICE: cuda:0
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_14b_cloud_advisor_20260611_105957.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_14b --target-detpass 90 --llm-mode worker --population 4 --gens 3 --min-generations 3 --max-generations 3 --sample-size 1 --validation-size 1 --cheap-eval-limit 2 --candidate-k 1 --repair-attempts 0 --det-profile strict --feedback-guided-mutation --selection-mode redesign --fitness-mode phase_aware --mu

KeyboardInterrupt: 

## 변경: 최신 strong compression / advisor parsing 확인용. 
## 이 셀을 먼저 실행한 뒤, [POST-FIX ADVISOR RESPONSE CHECK] 셀을 실행

In [ ]:
import inspect
print(inspect.signature(run_ga_all_categories))

In [20]:
# ============================================================
# RETRY ONLY: real cloud advisor staged compression smoke
# This creates advisor_retry_out
# ============================================================

from pathlib import Path
import json
import traceback
import pandas as pd
import numpy as np

assert "run_ga_all_categories" in globals(), "run_ga_all_categories is not defined. Run wrapper cell first."
assert "SERVER_PRESET" in globals(), "SERVER_PRESET is not defined. Run config cell first."

try:
    advisor_retry_out = run_ga_all_categories(
        model_key="qwen25_coder_14b",
        categories=(1,),
        limit_per_category=1,
        sample_size=1,
        validation_size=1,
        population=4,
        gens=3,
        target_detpass=90,
        base_prefix=f"smoke_compression_advisor_retry_{SERVER_PRESET}",
        use_advisor=True,
        full_run=True,
        progress="verbose",
        timeout_sec=1200,
        retries=0,

        advisor_trigger_mode="always",
        advisor_min_population_for_child=4,
        advisor_force_child_quota=True,
        use_mock_advisor=False,

        compression_detpass_threshold=90,
        aggressive_compression_after_target=True,

        compression_child_quota=1,
        compression_child_ratio=0.25,

        advisor_compression_child_quota=1,
        advisor_prefer_compression_after_detpass=90,

        compression_token_reduction_target=0.15,
        compression_token_plateau_delta=1.0,
        allow_aggressive_compression=True,

        micro_compression_child_quota=1,
        micro_compression_child_ratio=0.25,

        block_compression_child_quota=1,
        block_compression_child_ratio=0.25,

        multi_block_compression_child_quota=1,
        multi_block_compression_child_ratio=0.25,

        global_budget_compression_child_quota=0,

        enable_block_token_breakdown=True,
        enable_multi_block_compression=True,

        # 논문 primary smoke에서는 False가 안전함.
        enable_render_budget_compression=False,

        min_compression_token_delta=50,
    )

    advisor_retry_out = Path(advisor_retry_out)

    print("\n" + "=" * 120)
    print("[OK] advisor_retry_out created")
    print("advisor_retry_out =", advisor_retry_out)
    print("exists:", advisor_retry_out.exists())
    print("=" * 120)

except Exception as e:
    traceback.print_exc()
    raise

Traceback (most recent call last):
  File "/tmp/ipykernel_1993053/3813417374.py", line 16, in <module>
    advisor_retry_out = run_ga_all_categories(
TypeError: run_ga_all_categories() got an unexpected keyword argument 'micro_compression_child_quota'


TypeError: run_ga_all_categories() got an unexpected keyword argument 'micro_compression_child_quota'

In [ ]:
from pathlib import Path
import pandas as pd
import json

OUT_DIR = Path(advisor_retry_out)

print("=" * 120)
print("[NEW COMPRESSION ARTIFACT CHECK]")
print("=" * 120)

for name in [
    "block_token_breakdown.json",
    "prompt_token_breakdown.json",
    "mutation_proposals.jsonl",
    "advisor_mutation_proposals.jsonl",
    "population_transitions.csv",
    "ga_generation_progress.csv",
]:
    p = OUT_DIR / name
    print(f"{name:36s} {p.exists()}")

print("\n[Advisor prompt keyword check]")
for p in sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt")):
    text = p.read_text(encoding="utf-8", errors="replace")
    print("\n", p.name)
    for key in [
        "Advisor Case A",
        "Advisor Case B",
        "Advisor Case C",
        "block_token_breakdown",
        "prompt_token_breakdown",
        "block_compression_proposals",
        "multi_block_compression_proposals",
        "global_budget_compression_proposals",
    ]:
        print(f"{key:40s}", key in text)

print("\n[Population transition compression lanes]")
trans_path = OUT_DIR / "population_transitions.csv"
if trans_path.exists():
    trans = pd.read_csv(trans_path)
    cols = [
        "generation",
        "compression_state",
        "compression_ready",
        "aggressive_compression",
        "new_by_micro_compression",
        "new_by_block_compression",
        "new_by_multi_block_compression",
        "new_by_global_budget_compression",
        "new_by_compression",
        "new_by_advisor",
        "new_by_compression_fallback",
    ]
    cols = [c for c in cols if c in trans.columns]
    display(trans[cols])

print("\n[Mutation proposals: compression only]")
prop_path = OUT_DIR / "mutation_proposals.jsonl"
rows = []
if prop_path.exists():
    for line in prop_path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.strip():
            continue
        try:
            obj = json.loads(line)
        except Exception:
            continue
        if obj.get("mutation_family") == "compression" or "compress" in str(obj.get("operator", "")):
            rows.append(obj)

if rows:
    df = pd.DataFrame(rows)
    cols = [
        "generation",
        "source",
        "mutation_family",
        "compression_level",
        "operator",
        "target_block_id",
        "selected_block_ids",
        "expected_token_delta",
        "measured_prompt_token_delta",
        "rejection_reason",
    ]
    cols = [c for c in cols if c in df.columns]
    display(df[cols])
else:
    print("No compression proposal rows found.")

## 실행 결과 artifact 전체 구조 확인

In [ ]:
from pathlib import Path
import json
import ast
import pandas as pd
import numpy as np

# 방금 성공한 advisor retry 결과 경로
OUT_DIR = Path(advisor_retry_out)

print("=" * 120)
print("OUT_DIR:", OUT_DIR)
print("=" * 120)

files = [
    "ga_generation_progress.csv",
    "population_transitions.csv",
    "ga_block_diffs.jsonl",
    "advisor_feedback_batches.jsonl",
    "advisor_mutation_proposals.jsonl",
    "mutation_proposals.jsonl",
    "advisor_mutation_summary.csv",
    "ga_summary.json",
    "pareto_archive.csv",
]

for f in files:
    p = OUT_DIR / f
    print(f"{f:36s}: {p.exists()}")

print("\n[advisor prompt files]")
for p in sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt")):
    print("-", p.name, p.stat().st_size, "bytes")

print("\n[advisor response files]")
for p in sorted(OUT_DIR.glob("advisor_response_generation_*.json")):
    print("-", p.name, p.stat().st_size, "bytes")

print("\n[candidate csv files]")
cand_dir = OUT_DIR / "candidates"
if cand_dir.exists():
    csvs = sorted(cand_dir.glob("*.csv"))
    print("candidate csv count:", len(csvs))
    for p in csvs[:10]:
        print("-", p.name)
else:
    print("No candidates dir:", cand_dir)

In [ ]:
# ============================================================
# Inspect exact cloud advisor prompt
# ============================================================

advisor_prompts = sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt"))

if not advisor_prompts:
    print("[WARN] advisor_prompt_generation_*.txt not found.")
else:
    for p in advisor_prompts:
        text = p.read_text(encoding="utf-8", errors="replace")
        print("\n" + "=" * 120)
        print("ADVISOR PROMPT:", p.name)
        print("chars:", len(text))
        print("approx tokens by chars/4:", round(len(text) / 4))
        print("=" * 120)

        # 전체를 다 출력하면 너무 길 수 있으므로 핵심 keyword 주변만 출력
        keywords = [
            "compression_policy",
            "compression",
            "token",
            "DETPass",
            "candidate_strategies",
            "few_shot",
            "allowed_mutation_types",
            "required_response_schema",
            "compress_candidate_strategies_to_minimal",
            "reduce_few_shot_count",
            "lower_output_max_tokens",
            "drop_optional_blocks_for_budget",
        ]

        lines = text.splitlines()
        hit_lines = []

        for i, line in enumerate(lines):
            low = line.lower()
            if any(k.lower() in low for k in keywords):
                hit_lines.append(i)

        # 중복 주변 제거
        selected = set()
        for i in hit_lines:
            for j in range(max(0, i - 2), min(len(lines), i + 3)):
                selected.add(j)

        selected = sorted(selected)

        print("\n[KEY SECTIONS]")
        for i in selected[:250]:
            print(f"{i+1:04d}: {lines[i]}")

        if len(selected) > 250:
            print(f"\n... truncated {len(selected) - 250} selected lines")

        # 필요하면 전체 prompt를 별도 txt로 저장
        dump_path = OUT_DIR / f"INSPECT_FULL_{p.name}"
        dump_path.write_text(text, encoding="utf-8")
        print("\nFull advisor prompt copied to:", dump_path)

In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT_DIR = Path(OUT_DIR)

print("=" * 120)
print("[ARTIFACT EXISTS]")
print("=" * 120)

for name in [
    "block_token_breakdown.json",
    "prompt_token_breakdown.json",
    "mutation_proposals.jsonl",
    "advisor_mutation_proposals.jsonl",
    "population_transitions.csv",
    "ga_generation_progress.csv",
    "advisor_block_compression_plan_generation_001.json",
    "advisor_block_compression_plan_generation_002.json",
    "advisor_block_compression_plan_generation_003.json",
]:
    p = OUT_DIR / name
    print(f"{name:48s} {p.exists()}")

print("\n" + "=" * 120)
print("[GA PROGRESS]")
print("=" * 120)

progress = pd.read_csv(OUT_DIR / "ga_generation_progress.csv")
cols = [
    "generation",
    "validation_det_pass_rate",
    "validation_avg_det_score",
    "best_so_far_DETPass",
    "avg_prompt_tokens",
    "compression_phase",
    "compression_ready",
    "aggressive_compression",
]
cols = [c for c in cols if c in progress.columns]
display(progress[cols])

print("\n" + "=" * 120)
print("[POPULATION TRANSITIONS]")
print("=" * 120)

trans = pd.read_csv(OUT_DIR / "population_transitions.csv")
cols = [
    "generation",
    "compression_ready",
    "compression_phase",
    "new_by_compression",
    "new_by_micro_compression",
    "new_by_block_compression",
    "new_by_multi_block_compression",
    "new_by_global_budget_compression",
    "new_by_compression_fallback",
    "new_by_advisor",
]
cols = [c for c in cols if c in trans.columns]
display(trans[cols])

print("\n" + "=" * 120)
print("[COMPRESSION PROPOSALS]")
print("=" * 120)

rows = []
for path_name in ["mutation_proposals.jsonl", "advisor_mutation_proposals.jsonl"]:
    p = OUT_DIR / path_name
    if not p.exists():
        continue

    for line in p.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.strip():
            continue
        try:
            obj = json.loads(line)
        except Exception:
            continue

        if (
            obj.get("mutation_family") == "compression"
            or "compress" in str(obj.get("operator", ""))
            or "few_shot" in str(obj.get("operator", ""))
            or "token" in str(obj.get("operator", ""))
        ):
            obj["_file"] = path_name
            rows.append(obj)

df = pd.DataFrame(rows)
if len(df):
    cols = [
        "_file",
        "generation",
        "source",
        "mutation_family",
        "compression_level",
        "operator",
        "mutation_type",
        "target_block_id",
        "selected_block_id",
        "selected_block_ids",
        "expected_token_delta",
        "measured_prompt_token_delta",
        "rejection_reason",
    ]
    cols = [c for c in cols if c in df.columns]
    display(df[cols])
else:
    print("No compression proposals found.")

In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT_DIR = Path(advisor_retry_out)

print("=" * 120)
print("[POST-FIX ADVISOR RESPONSE CHECK]")
print("=" * 120)

for p in sorted(OUT_DIR.glob("advisor_response_generation_*.json")):
    print("\n" + "-" * 120)
    print(p.name, p.stat().st_size, "bytes")

    obj = json.loads(p.read_text(encoding="utf-8", errors="replace"))

    print("top keys:", list(obj.keys()))
    parsed = obj.get("parsed", {})
    print("parsed type:", type(parsed).__name__)

    if isinstance(parsed, dict):
        print("parsed keys:", list(parsed.keys()))

        for key in [
            "advisor_status",
            "compression_policy",
            "proposals",
            "micro_compression_proposals",
            "block_compression_proposals",
            "multi_block_compression_proposals",
            "global_budget_compression_proposals",
            "prompt_token_breakdown_seen",
            "block_token_breakdown_seen",
        ]:
            v = parsed.get(key)
            if isinstance(v, list):
                print(f"{key}: list len={len(v)}")
            else:
                print(f"{key}: {v is not None}")

    print("accepted_proposals:", len(obj.get("accepted_proposals", []) or []))
    print("rejected_proposals:", len(obj.get("rejected_proposals", []) or []))

    cp = obj.get("compression_policy", None)
    print("top-level compression_policy exists:", isinstance(cp, dict))

In [ ]:
print("=" * 120)
print("[ADVISOR MUTATION SUMMARY]")
print("=" * 120)

summary_path = OUT_DIR / "advisor_mutation_summary.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary)
    print("rows:", len(summary))
else:
    print("missing:", summary_path)

print("=" * 120)
print("[ADVISOR MUTATION PROPOSALS JSONL]")
print("=" * 120)

proposal_path = OUT_DIR / "advisor_mutation_proposals.jsonl"
rows = []
if proposal_path.exists():
    for line in proposal_path.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.strip():
            rows.append(json.loads(line))

print("rows:", len(rows))

if rows:
    df = pd.DataFrame(rows)
    cols = [
        "generation",
        "source",
        "advisor_batch_id",
        "proposal_id",
        "schema_source",
        "mutation_family",
        "compression_level",
        "operator",
        "mutation_type",
        "target_block_id",
        "selected_block_id",
        "selected_block_ids",
        "target_block_family",
        "expected_token_delta",
        "measured_prompt_token_delta",
        "accepted",
        "rejection_reason",
        "raw_response_path",
        "advisor_prompt_path",
    ]
    cols = [c for c in cols if c in df.columns]
    display(df[cols])
else:
    print("No advisor proposal rows.")

In [ ]:
print("=" * 120)
print("[ADVISOR BLOCK PLAN CHECK]")
print("=" * 120)

for p in sorted(OUT_DIR.glob("advisor_block_compression_plan_generation_*.json")):
    print("\n" + "-" * 120)
    print(p.name)

    obj = json.loads(p.read_text(encoding="utf-8", errors="replace"))
    print("keys:", list(obj.keys()))

    for key in [
        "accepted_block_proposals",
        "accepted_multi_block_proposals",
        "accepted_global_budget_proposals",
        "rejected_compression_proposals",
    ]:
        v = obj.get(key, [])
        print(f"{key}: {len(v) if isinstance(v, list) else type(v).__name__}")

    print("fallback_used:", obj.get("fallback_used"))
    print("fallback_reason:", obj.get("fallback_reason"))

In [ ]:
print("=" * 120)
print("[TOKEN REDUCTION CHECK]")
print("=" * 120)

progress = pd.read_csv(OUT_DIR / "ga_generation_progress.csv")
cols = [
    "generation",
    "validation_det_pass_rate",
    "validation_avg_det_score",
    "best_so_far_DETPass",
    "avg_prompt_tokens",
    "compression_phase",
    "compression_ready",
    "aggressive_compression",
]
cols = [c for c in cols if c in progress.columns]
display(progress[cols])

tokens = pd.to_numeric(progress["avg_prompt_tokens"], errors="coerce")
print("first tokens:", tokens.iloc[0])
print("last tokens:", tokens.iloc[-1])
print("delta:", tokens.iloc[-1] - tokens.iloc[0])
print("reduction ratio:", (tokens.iloc[0] - tokens.iloc[-1]) / tokens.iloc[0] if tokens.iloc[0] else None)

In [ ]:
# ============================================================
# FINAL FULL FAIR RUN
# cloudless vs cloud_advisor with staged strong compression
# ============================================================

from pathlib import Path
import os
import json
import time
import traceback
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 0. Safety checks
# ------------------------------------------------------------

assert "run_ga_all_categories" in globals(), "run_ga_all_categories is not defined."
assert "SERVER_PRESET" in globals(), "SERVER_PRESET is not defined."

if not os.environ.get("OPENAI_API_KEY", "").strip():
    raise RuntimeError("OPENAI_API_KEY is not set. Cloud advisor full run requires it.")

# ------------------------------------------------------------
# 1. Models / modes
# ------------------------------------------------------------

MODEL_LIST = [
    ("7B", "qwen25_coder_7b"),
    ("8B", "llama31_8b"),
    ("14B", "qwen25_coder_14b"),
]

RUN_MODES = [
    ("cloudless", False),
    ("cloud_advisor", True),
]

# ------------------------------------------------------------
# 2. Full GA config
# ------------------------------------------------------------

COMMON_GA_CONFIG = {
    "categories": tuple(range(1, 9)),
    "limit_per_category": 3,
    "sample_size": 24,
    "validation_size": 24,
    "population": 5,
    "gens": 10,
    "target_detpass": 90,
    "full_run": True,
    "progress": "verbose",
    "timeout_sec": 1800,
    "retries": 0,
    "idle_timeout_sec": None,
    "total_timeout_sec": None,
}

# ------------------------------------------------------------
# 3. Strong compression config
# ------------------------------------------------------------

COMMON_COMPRESSION_KWARGS = dict(
    compression_detpass_threshold=90,
    aggressive_compression_after_target=True,

    compression_child_quota=1,
    compression_child_ratio=0.25,

    advisor_compression_child_quota=1,
    advisor_prefer_compression_after_detpass=90,

    compression_token_reduction_target=0.15,
    compression_token_plateau_delta=1.0,
    allow_aggressive_compression=True,

    micro_compression_child_quota=1,
    micro_compression_child_ratio=0.25,

    block_compression_child_quota=1,
    block_compression_child_ratio=0.25,

    multi_block_compression_child_quota=1,
    multi_block_compression_child_ratio=0.25,

    global_budget_compression_child_quota=0,

    enable_block_token_breakdown=True,
    enable_multi_block_compression=True,

    # renderer 연결/효과가 아직 불확실하면 False가 논문용 primary run에 더 안전함.
    enable_render_budget_compression=False,

    min_compression_token_delta=50,
)

# ------------------------------------------------------------
# 4. Lightweight artifact summary
# ------------------------------------------------------------

def summarize_full_run(out_dir):
    out_dir = Path(out_dir)

    row = {
        "out_dir": str(out_dir),
        "summary_exists": (out_dir / "ga_summary.json").exists(),
        "progress_exists": (out_dir / "ga_generation_progress.csv").exists(),
        "transition_exists": (out_dir / "population_transitions.csv").exists(),
        "mutation_proposals_exists": (out_dir / "mutation_proposals.jsonl").exists(),
        "advisor_proposals_exists": (out_dir / "advisor_mutation_proposals.jsonl").exists(),
        "block_breakdown_exists": (out_dir / "block_token_breakdown.json").exists(),
        "prompt_breakdown_exists": (out_dir / "prompt_token_breakdown.json").exists(),
        "best_DETPass": np.nan,
        "best_so_far_DETPass": np.nan,
        "first_avg_prompt_tokens": np.nan,
        "last_avg_prompt_tokens": np.nan,
        "token_delta": np.nan,
        "token_reduction_ratio": np.nan,
        "new_by_micro_compression": 0,
        "new_by_block_compression": 0,
        "new_by_multi_block_compression": 0,
        "new_by_global_budget_compression": 0,
        "new_by_compression_fallback": 0,
        "advisor_proposal_rows": 0,
        "advisor_accepted_rows": 0,
        "advisor_rejected_rows": 0,
    }

    summary_path = out_dir / "ga_summary.json"
    if summary_path.exists():
        try:
            s = json.loads(summary_path.read_text(encoding="utf-8", errors="replace"))
            row["best_DETPass"] = s.get("best_DETPass", s.get("best_so_far_DETPass", np.nan))
            row["best_so_far_DETPass"] = s.get("best_so_far_DETPass", np.nan)
        except Exception:
            pass

    progress_path = out_dir / "ga_generation_progress.csv"
    if progress_path.exists():
        try:
            progress = pd.read_csv(progress_path)
            if len(progress) and "avg_prompt_tokens" in progress.columns:
                tokens = pd.to_numeric(progress["avg_prompt_tokens"], errors="coerce")
                row["first_avg_prompt_tokens"] = float(tokens.iloc[0])
                row["last_avg_prompt_tokens"] = float(tokens.iloc[-1])
                row["token_delta"] = row["last_avg_prompt_tokens"] - row["first_avg_prompt_tokens"]
                if row["first_avg_prompt_tokens"]:
                    row["token_reduction_ratio"] = (
                        row["first_avg_prompt_tokens"] - row["last_avg_prompt_tokens"]
                    ) / row["first_avg_prompt_tokens"]
            if len(progress) and "best_so_far_DETPass" in progress.columns:
                row["best_so_far_DETPass"] = float(
                    pd.to_numeric(progress["best_so_far_DETPass"], errors="coerce").max()
                )
        except Exception:
            pass

    trans_path = out_dir / "population_transitions.csv"
    if trans_path.exists():
        try:
            trans = pd.read_csv(trans_path)
            for col in [
                "new_by_micro_compression",
                "new_by_block_compression",
                "new_by_multi_block_compression",
                "new_by_global_budget_compression",
                "new_by_compression_fallback",
            ]:
                if col in trans.columns:
                    row[col] = int(pd.to_numeric(trans[col], errors="coerce").fillna(0).sum())
        except Exception:
            pass

    advisor_jsonl = out_dir / "advisor_mutation_proposals.jsonl"
    if advisor_jsonl.exists():
        rows = []
        for line in advisor_jsonl.read_text(encoding="utf-8", errors="replace").splitlines():
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                pass

        row["advisor_proposal_rows"] = len(rows)
        row["advisor_accepted_rows"] = sum(1 for r in rows if r.get("accepted") is True)
        row["advisor_rejected_rows"] = sum(1 for r in rows if r.get("accepted") is False)

    return row

# ------------------------------------------------------------
# 5. Final full fair loop
# ------------------------------------------------------------

ga_runs_fair = {}
ga_summary_rows = []

for label, model_key in MODEL_LIST:
    for mode_name, use_advisor in RUN_MODES:
        run_key = f"{label}_{mode_name}"

        print("\n" + "#" * 120)
        print(f"START RUN: {run_key} / {model_key}")
        print("#" * 120)

        try:
            out_dir = run_ga_all_categories(
                model_key=model_key,
                categories=COMMON_GA_CONFIG["categories"],
                limit_per_category=COMMON_GA_CONFIG["limit_per_category"],
                sample_size=COMMON_GA_CONFIG["sample_size"],
                validation_size=COMMON_GA_CONFIG["validation_size"],
                population=COMMON_GA_CONFIG["population"],
                gens=COMMON_GA_CONFIG["gens"],
                target_detpass=COMMON_GA_CONFIG["target_detpass"],
                base_prefix=f"ga_{SERVER_PRESET}_compression",
                use_advisor=use_advisor,
                full_run=COMMON_GA_CONFIG["full_run"],
                progress=COMMON_GA_CONFIG["progress"],
                timeout_sec=COMMON_GA_CONFIG["timeout_sec"],
                retries=COMMON_GA_CONFIG["retries"],
                idle_timeout_sec=COMMON_GA_CONFIG["idle_timeout_sec"],
                total_timeout_sec=COMMON_GA_CONFIG["total_timeout_sec"],

                advisor_trigger_mode="always" if use_advisor else "off",
                advisor_min_population_for_child=4,
                advisor_force_child_quota=True if use_advisor else False,
                use_mock_advisor=False,

                **COMMON_COMPRESSION_KWARGS,
            )

            ga_runs_fair[run_key] = out_dir
            row = summarize_full_run(out_dir)
            row["run_key"] = run_key
            row["model_key"] = model_key
            row["mode"] = mode_name
            ga_summary_rows.append(row)

            print(f"[PASS] {run_key}: {out_dir}")
            print("[SUMMARY]", row)

        except Exception as e:
            ga_runs_fair[run_key] = None
            err_row = {
                "run_key": run_key,
                "model_key": model_key,
                "mode": mode_name,
                "out_dir": None,
                "error": repr(e),
            }
            ga_summary_rows.append(err_row)

            print(f"[FAIL] {run_key}: {type(e).__name__}: {e}")
            traceback.print_exc()

        time.sleep(5)

# ------------------------------------------------------------
# 6. Final report
# ------------------------------------------------------------

valid_ga_runs_fair = {
    k: v for k, v in ga_runs_fair.items()
    if v is not None
}

print("\n" + "=" * 120)
print("VALID RUNS")
print("=" * 120)
for k, v in valid_ga_runs_fair.items():
    print(k, "=>", v)

summary_df = pd.DataFrame(ga_summary_rows)
display(summary_df)

if len(valid_ga_runs_fair) != len(MODEL_LIST) * len(RUN_MODES):
    print("[WARN] Some runs failed.")
else:
    print("[OK] All full fair runs completed.")

valid_ga_runs_fair

In [ ]:
enable_multi_block_compression=True
enable_block_token_breakdown=True
enable_render_budget_compression=True
block_compression_child_quota=1
multi_block_compression_child_quota=1
micro_compression_child_quota=1

out = run_ga_all_categories(
    model_key="qwen25_coder_14b",
    categories=(3, 4, 5, 6),
    limit_per_category=1,
    sample_size=4,
    validation_size=4,
    population=5,
    gens=5,
    target_detpass=90,
    base_prefix=f"smoke_strong_compression_{SERVER_PRESET}",
    use_advisor=True,
    full_run=True,
    progress="verbose",
    timeout_sec=1800,
    retries=0,

    advisor_trigger_mode="always",
    advisor_min_population_for_child=4,
    advisor_force_child_quota=True,
    use_mock_advisor=False,

    compression_detpass_threshold=90,
    aggressive_compression_after_target=True,
    allow_aggressive_compression=True,
)

# Cell 3. 최종 full fair run 실행

In [ ]:
# ============================================================
# Final full fair run: cloudless vs cloud_advisor with compression
# ============================================================

MODEL_LIST = [
    ("7B", "qwen25_coder_7b"),
    ("8B", "llama31_8b"),
    ("14B", "qwen25_coder_14b"),
]

RUN_MODES = [
    ("cloudless", False),
    ("cloud_advisor", True),
]

COMMON_GA_CONFIG = {
    "categories": tuple(range(1, 9)),
    "limit_per_category": 3,
    "sample_size": 24,
    "validation_size": 24,
    "population": 5,
    "gens": 10,
    "target_detpass": 90,
    "full_run": True,
    "progress": "verbose",
    "timeout_sec": 1200,
    "retries": 0,
    "idle_timeout_sec": None,
    "total_timeout_sec": None,
}

ga_runs_fair = {}

for label, model_key in MODEL_LIST:
    for mode_name, use_advisor in RUN_MODES:
        run_key = f"{label}_{mode_name}"

        print("\n" + "#" * 120)
        print(f"START RUN: {run_key} / {model_key}")
        print("#" * 120)

        try:
            out_dir = run_ga_all_categories(
                model_key=model_key,
                categories=COMMON_GA_CONFIG["categories"],
                limit_per_category=COMMON_GA_CONFIG["limit_per_category"],
                sample_size=COMMON_GA_CONFIG["sample_size"],
                validation_size=COMMON_GA_CONFIG["validation_size"],
                population=COMMON_GA_CONFIG["population"],
                gens=COMMON_GA_CONFIG["gens"],
                target_detpass=COMMON_GA_CONFIG["target_detpass"],
                base_prefix=f"ga_{SERVER_PRESET}_compression",
                use_advisor=use_advisor,
                full_run=COMMON_GA_CONFIG["full_run"],
                progress=COMMON_GA_CONFIG["progress"],
                timeout_sec=COMMON_GA_CONFIG["timeout_sec"],
                retries=COMMON_GA_CONFIG["retries"],
                idle_timeout_sec=COMMON_GA_CONFIG["idle_timeout_sec"],
                total_timeout_sec=COMMON_GA_CONFIG["total_timeout_sec"],

                advisor_trigger_mode="always" if use_advisor else "off",
                advisor_min_population_for_child=4,
                advisor_force_child_quota=True if use_advisor else False,
                use_mock_advisor=False,

                **COMMON_COMPRESSION_KWARGS,
            )

            ga_runs_fair[run_key] = out_dir
            print(f"[PASS] {run_key}: {out_dir}")

        except Exception as e:
            ga_runs_fair[run_key] = None
            print(f"[FAIL] {run_key}: {type(e).__name__}: {e}")
            traceback.print_exc()

        time.sleep(5)

valid_ga_runs_fair = {k: v for k, v in ga_runs_fair.items() if v is not None}

print("\n" + "=" * 120)
print("VALID RUNS")
print("=" * 120)
for k, v in valid_ga_runs_fair.items():
    print(k, "=>", v)

if len(valid_ga_runs_fair) != len(MODEL_LIST) * len(RUN_MODES):
    print("[WARN] Some runs failed.")
else:
    print("[OK] All full fair runs completed.")

valid_ga_runs_fair

# Cell 4. 결과 요약 테이블 생성

In [ ]:
# ============================================================
# Build paper-ready summary tables from ga_runs_fair
# ============================================================

def read_json(path):
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {}

def read_progress(out_dir):
    p = Path(out_dir) / "ga_generation_progress.csv"
    return pd.read_csv(p) if p.exists() else pd.DataFrame()

def read_transitions(out_dir):
    p = Path(out_dir) / "population_transitions.csv"
    return pd.read_csv(p) if p.exists() else pd.DataFrame()

def summarize_run(run_key, out_dir):
    out_dir = Path(out_dir)
    label, mode = run_key.split("_", 1)

    s = read_json(out_dir / "ga_summary.json")
    progress = read_progress(out_dir)
    trans = read_transitions(out_dir)
    proposals_path = out_dir / "mutation_proposals.jsonl"

    row = {
        "run_key": run_key,
        "label": label,
        "mode": mode,
        "out_dir": str(out_dir),
        "summary_exists": (out_dir / "ga_summary.json").exists(),
        "progress_exists": (out_dir / "ga_generation_progress.csv").exists(),
        "best_generation": s.get("best_generation"),
        "best_DETPass": s.get("best_DETPass", s.get("best_so_far_DETPass")),
        "best_avg_DET": s.get("best_avg_DET"),
        "best_so_far_DETPass": s.get("best_so_far_DETPass"),
        "accepted_best_DETPass": s.get("accepted_best_DETPass"),
        "accepted_best_tokens": s.get("accepted_best_tokens", s.get("compact_best_tokens")),
        "compact_best_tokens": s.get("compact_best_tokens"),
        "compact_best_DETPass": s.get("compact_best_DETPass"),
        "pareto_archive_size": s.get("pareto_archive_size"),
        "stop_reason": s.get("stop_reason"),
        "final_phase": s.get("final_phase"),
        "compression_ready_final": s.get("compression_ready_final"),
        "compression_success_count": s.get("compression_success_count", 0),
        "compression_rejection_count": s.get("compression_rejection_count", 0),
        "advisor_compression_proposals_generated": s.get("advisor_compression_proposals_generated", 0),
        "advisor_compression_children_scheduled": s.get("advisor_compression_children_scheduled", 0),
        "cloudless_compression_fallback_scheduled": s.get("cloudless_compression_fallback_scheduled", 0),
        "advisor_proposals_generated": s.get("advisor_proposals_generated", 0),
        "advisor_children_scheduled": s.get("advisor_children_scheduled", 0),
    }

    if len(progress):
        for c in ["avg_prompt_tokens", "validation_det_pass_rate", "validation_avg_det_score"]:
            if c in progress.columns:
                row[c] = pd.to_numeric(progress[c], errors="coerce").iloc[-1]

    if len(trans):
        for c in [
            "new_by_compression",
            "new_by_advisor",
            "advisor_compression_proposals",
            "advisor_compression_children_scheduled",
            "cloudless_compression_fallback_scheduled",
        ]:
            if c in trans.columns:
                row[f"{c}_sum"] = int(pd.to_numeric(trans[c], errors="coerce").fillna(0).sum())

    row["compression_proposal_count"] = 0
    row["advisor_proposal_count"] = 0
    if proposals_path.exists():
        for line in proposals_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue
            source = str(obj.get("source", ""))
            family = str(obj.get("mutation_family", ""))
            op = str(obj.get("operator", ""))
            if source == "advisor":
                row["advisor_proposal_count"] += 1
            if family == "compression" or "compress" in op or "token" in op or "few_shot" in op:
                row["compression_proposal_count"] += 1

    token_value = row.get("accepted_best_tokens")
    if token_value is None or pd.isna(token_value):
        token_value = row.get("avg_prompt_tokens", np.nan)

    row["tokens_gt_0"] = pd.to_numeric(pd.Series([token_value]), errors="coerce").fillna(0).iloc[0] > 0
    row["detpass_gt_0"] = pd.to_numeric(pd.Series([row.get("best_DETPass")]), errors="coerce").fillna(0).iloc[0] > 0
    row["run_ok"] = bool(row["summary_exists"] and row["progress_exists"] and row["tokens_gt_0"] and row["detpass_gt_0"])

    return row


summary_rows = [
    summarize_run(k, v)
    for k, v in valid_ga_runs_fair.items()
]

fair_summary = pd.DataFrame(summary_rows)
display(fair_summary)

# ------------------------------------------------------------
# Paper main table
# ------------------------------------------------------------

model_map = {
    "7B": "Qwen2.5-Coder-7B",
    "8B": "Llama-3.1-8B",
    "14B": "Qwen2.5-Coder-14B",
}

config_map = {
    "cloudless": "GPS-PromptOps",
    "cloud_advisor": "GPS-PromptOps + Cloud Advisor",
}

paper_table = pd.DataFrame({
    "Model": fair_summary["label"].map(model_map).fillna(fair_summary["label"]),
    "Configuration": fair_summary["mode"].map(config_map).fillna(fair_summary["mode"]),
    "DETPass (%)": pd.to_numeric(fair_summary["best_DETPass"], errors="coerce"),
    "Avg S_DET": pd.to_numeric(fair_summary["best_avg_DET"], errors="coerce") / 100.0,
    "Avg Input Tokens": pd.to_numeric(
        fair_summary["accepted_best_tokens"].fillna(fair_summary.get("avg_prompt_tokens")),
        errors="coerce",
    ),
    "Compact Tokens": pd.to_numeric(fair_summary["compact_best_tokens"], errors="coerce"),
    "Pareto Archive": pd.to_numeric(fair_summary["pareto_archive_size"], errors="coerce"),
    "Compression Proposals": pd.to_numeric(fair_summary["compression_proposal_count"], errors="coerce"),
    "New Compression Children": pd.to_numeric(fair_summary.get("new_by_compression_sum", 0), errors="coerce"),
    "Advisor Compression Children": pd.to_numeric(
        fair_summary.get("advisor_compression_children_scheduled_sum", fair_summary["advisor_compression_children_scheduled"]),
        errors="coerce",
    ),
    "Fallback Compression": pd.to_numeric(
        fair_summary.get("cloudless_compression_fallback_scheduled_sum", fair_summary["cloudless_compression_fallback_scheduled"]),
        errors="coerce",
    ),
    "Run OK": fair_summary["run_ok"],
})

avg_rows = []
for cfg, sub in paper_table.groupby("Configuration"):
    avg_rows.append({
        "Model": "Target-model average",
        "Configuration": cfg,
        "DETPass (%)": sub["DETPass (%)"].mean(),
        "Avg S_DET": sub["Avg S_DET"].mean(),
        "Avg Input Tokens": sub["Avg Input Tokens"].mean(),
        "Compact Tokens": sub["Compact Tokens"].mean(),
        "Pareto Archive": sub["Pareto Archive"].mean(),
        "Compression Proposals": sub["Compression Proposals"].sum(),
        "New Compression Children": sub["New Compression Children"].sum(),
        "Advisor Compression Children": sub["Advisor Compression Children"].sum(),
        "Fallback Compression": sub["Fallback Compression"].sum(),
        "Run OK": sub["Run OK"].all(),
    })

paper_table_final = pd.concat([paper_table, pd.DataFrame(avg_rows)], ignore_index=True)

display(
    paper_table_final.style.format({
        "DETPass (%)": "{:.2f}",
        "Avg S_DET": "{:.4f}",
        "Avg Input Tokens": "{:.1f}",
        "Compact Tokens": "{:.1f}",
        "Pareto Archive": "{:.0f}",
        "Compression Proposals": "{:.0f}",
        "New Compression Children": "{:.0f}",
        "Advisor Compression Children": "{:.0f}",
        "Fallback Compression": "{:.0f}",
    }, na_rep="-")
)

# ------------------------------------------------------------
# Cloudless vs advisor delta
# ------------------------------------------------------------

delta_rows = []
for label in sorted(fair_summary["label"].unique()):
    cl = fair_summary[(fair_summary["label"] == label) & (fair_summary["mode"] == "cloudless")]
    ad = fair_summary[(fair_summary["label"] == label) & (fair_summary["mode"] == "cloud_advisor")]

    if len(cl) == 0 or len(ad) == 0:
        continue

    cl = cl.iloc[0]
    ad = ad.iloc[0]

    cl_tokens = pd.to_numeric(pd.Series([cl.get("accepted_best_tokens", cl.get("avg_prompt_tokens"))]), errors="coerce").iloc[0]
    ad_tokens = pd.to_numeric(pd.Series([ad.get("accepted_best_tokens", ad.get("avg_prompt_tokens"))]), errors="coerce").iloc[0]

    delta_rows.append({
        "label": label,
        "cloudless_DETPass": cl.get("best_DETPass"),
        "advisor_DETPass": ad.get("best_DETPass"),
        "delta_DETPass": ad.get("best_DETPass") - cl.get("best_DETPass"),
        "cloudless_tokens": cl_tokens,
        "advisor_tokens": ad_tokens,
        "delta_tokens": ad_tokens - cl_tokens,
        "cloudless_compression_proposals": cl.get("compression_proposal_count", 0),
        "advisor_compression_proposals": ad.get("compression_proposal_count", 0),
        "advisor_compression_children": ad.get("advisor_compression_children_scheduled_sum", ad.get("advisor_compression_children_scheduled", 0)),
        "fallback_compression_children": ad.get("cloudless_compression_fallback_scheduled_sum", ad.get("cloudless_compression_fallback_scheduled", 0)),
    })

delta_table = pd.DataFrame(delta_rows)
display(delta_table)

# Cell 5. Pareto 그래프 + Token/Compression 그래프

In [ ]:
# ============================================================
# Load Pareto points
# ============================================================

def load_pareto_points_from_out(run_key, out_dir):
    out_dir = Path(out_dir)
    label, mode = run_key.split("_", 1)

    candidates = [
        out_dir / "pareto_archive.csv",
        out_dir / "ga_pareto_frontier.csv",
        out_dir / "ga_pareto_generation_summary.csv",
    ]

    for p in candidates:
        if p.exists():
            df = pd.read_csv(p)
            df["run_key"] = run_key
            df["label"] = label
            df["mode"] = mode
            df["source_file"] = str(p)

            if "det_pass_rate" not in df.columns:
                for c in ["DETPass", "best_DETPass", "validation_det_pass_rate", "det"]:
                    if c in df.columns:
                        df["det_pass_rate"] = df[c]
                        break

            if "avg_prompt_tokens" not in df.columns:
                for c in ["accepted_best_tokens", "avg_tokens", "tokens", "prompt_tokens", "avg_prompt_tokens"]:
                    if c in df.columns:
                        df["avg_prompt_tokens"] = df[c]
                        break

            return df

    row = fair_summary[fair_summary["run_key"] == run_key].iloc[0]
    return pd.DataFrame([{
        "run_key": run_key,
        "label": label,
        "mode": mode,
        "generation": row.get("best_generation", np.nan),
        "genome_id": "",
        "det_pass_rate": row.get("best_DETPass", np.nan),
        "avg_prompt_tokens": row.get("accepted_best_tokens", row.get("avg_prompt_tokens", np.nan)),
        "source_file": "summary_fallback",
    }])


pareto_all = pd.concat(
    [load_pareto_points_from_out(k, v) for k, v in valid_ga_runs_fair.items()],
    ignore_index=True,
)

pareto_all["det_pass_rate"] = pd.to_numeric(pareto_all["det_pass_rate"], errors="coerce")
pareto_all["avg_prompt_tokens"] = pd.to_numeric(pareto_all["avg_prompt_tokens"], errors="coerce")
pareto_all = pareto_all.dropna(subset=["det_pass_rate", "avg_prompt_tokens"])

def global_pareto(df):
    d = df.sort_values(["avg_prompt_tokens", "det_pass_rate"], ascending=[True, False]).copy()
    best = -np.inf
    keep = []
    for idx, row in d.iterrows():
        if row["det_pass_rate"] > best:
            keep.append(idx)
            best = row["det_pass_rate"]
    d["is_pareto"] = d.index.isin(keep)
    return d

pareto_eval = global_pareto(pareto_all)
frontier = pareto_eval[pareto_eval["is_pareto"]].copy()

display(frontier[[
    "run_key", "label", "mode", "generation", "genome_id",
    "det_pass_rate", "avg_prompt_tokens", "source_file"
]])

# ============================================================
# Figure 1: DETPass vs tokens Pareto
# ============================================================

plt.figure(figsize=(9, 6))

for (label, mode), sub in pareto_eval.groupby(["label", "mode"]):
    plt.scatter(
        sub["avg_prompt_tokens"],
        sub["det_pass_rate"],
        s=70,
        alpha=0.75,
        label=f"{label}-{mode}",
    )

if len(frontier):
    f = frontier.sort_values("avg_prompt_tokens")
    plt.plot(
        f["avg_prompt_tokens"],
        f["det_pass_rate"],
        linestyle="--",
        linewidth=2,
        label="Global Pareto frontier",
    )

for _, r in fair_summary.iterrows():
    x = pd.to_numeric(pd.Series([r.get("accepted_best_tokens", r.get("avg_prompt_tokens"))]), errors="coerce").iloc[0]
    y = pd.to_numeric(pd.Series([r.get("best_DETPass")]), errors="coerce").iloc[0]
    if pd.notna(x) and pd.notna(y):
        plt.annotate(
            f"{r['label']}-{r['mode']}",
            (x, y),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=9,
        )

plt.xlabel("Average Input Tokens")
plt.ylabel("DETPass (%)")
plt.title("Deployment-aware Pareto Trade-off")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ============================================================
# Figure 2: advisor vs cloudless token delta
# ============================================================

if len(delta_table):
    plt.figure(figsize=(7, 4))
    plt.bar(delta_table["label"], delta_table["delta_tokens"])
    plt.axhline(0, linewidth=1)
    plt.xlabel("Model")
    plt.ylabel("Advisor - Cloudless Tokens")
    plt.title("Token Reduction by Cloud Advisor Compression")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.bar(delta_table["label"], delta_table["delta_DETPass"])
    plt.axhline(0, linewidth=1)
    plt.xlabel("Model")
    plt.ylabel("Advisor - Cloudless DETPass (%p)")
    plt.title("Accuracy Change by Cloud Advisor Compression")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

# ============================================================
# Figure 3: compression activity
# ============================================================

activity = fair_summary.copy()
activity["compression_total"] = (
    pd.to_numeric(activity.get("compression_proposal_count", 0), errors="coerce").fillna(0)
    + pd.to_numeric(activity.get("new_by_compression_sum", 0), errors="coerce").fillna(0)
    + pd.to_numeric(activity.get("advisor_compression_children_scheduled_sum", activity.get("advisor_compression_children_scheduled", 0)), errors="coerce").fillna(0)
    + pd.to_numeric(activity.get("cloudless_compression_fallback_scheduled_sum", activity.get("cloudless_compression_fallback_scheduled", 0)), errors="coerce").fillna(0)
)

plt.figure(figsize=(9, 4))
plt.bar(activity["run_key"], activity["compression_total"])
plt.xticks(rotation=45, ha="right")
plt.xlabel("Run")
plt.ylabel("Compression Activity Count")
plt.title("Compression Mutation / Advisor Scheduling Activity")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Cell 6. 논문 삽입용 CSV/LaTeX 저장

In [ ]:
# ============================================================
# Save paper-ready outputs
# ============================================================

OUT = RESULTS_ROOT / f"paper_outputs_{SERVER_PRESET}_compression"
OUT.mkdir(parents=True, exist_ok=True)

fair_summary.to_csv(OUT / "fair_summary_compression.csv", index=False)
paper_table_final.to_csv(OUT / "table_main_results_compression.csv", index=False)
delta_table.to_csv(OUT / "table_cloudless_vs_advisor_delta_compression.csv", index=False)
pareto_eval.to_csv(OUT / "pareto_points_all.csv", index=False)
frontier.to_csv(OUT / "pareto_frontier.csv", index=False)

paper_table_final.to_latex(
    OUT / "table_main_results_compression.tex",
    index=False,
    float_format="%.3f",
    na_rep="-",
    caption="Compression-aware prompt optimization results.",
    label="tab:compression_promptops_results",
)

delta_table.to_latex(
    OUT / "table_cloudless_vs_advisor_delta_compression.tex",
    index=False,
    float_format="%.3f",
    na_rep="-",
    caption="Effect of cloud-advisor compression feedback over cloudless GA feedback.",
    label="tab:cloud_advisor_compression_delta",
)

print("Saved to:", OUT)
for p in sorted(OUT.iterdir()):
    print("-", p)

print("\n[FINAL CHECK]")
print("All runs:", len(ga_runs_fair))
print("Valid runs:", len(valid_ga_runs_fair))
print("Run OK count:", int(fair_summary["run_ok"].sum()), "/", len(fair_summary))
print("Compression proposal total:", int(pd.to_numeric(fair_summary["compression_proposal_count"], errors="coerce").fillna(0).sum()))
print("Advisor compression children total:", int(pd.to_numeric(fair_summary.get("advisor_compression_children_scheduled_sum", fair_summary["advisor_compression_children_scheduled"]), errors="coerce").fillna(0).sum()))
print("Cloudless compression fallback total:", int(pd.to_numeric(fair_summary.get("cloudless_compression_fallback_scheduled_sum", fair_summary["cloudless_compression_fallback_scheduled"]), errors="coerce").fillna(0).sum()))